# 🎗️ Predicción de Cáncer con Random Forest y XGBoost
## Taller Pre-Congreso CNIB 2025

---

> **Contexto:** Este notebook usa un dataset de Kaggle con 1500 registros de pacientes para entrenar modelos de predicción de cáncer. Aplicamos dos algoritmos de ensemble de alto rendimiento: **Random Forest** y **XGBoost**, y luego los interpretamos con **SHAP** y **LIME**.

---


In [ ]:
# Instalamos las librerías necesarias
!pip install lime
!pip install imbalanced-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import numpy as np
%matplotlib inline
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)
import shap
import lightgbm as lgb

## 📑 Dataset: Cancer Prediction (Kaggle)

El dataset contiene variables de riesgo conocidas para cáncer:
- Variables demográficas (edad, sexo)
- Hábitos de vida (tabaquismo, alcohol, sedentarismo)
- Antecedentes genéticos y familiares
- Exposición a agentes carcinógenos
- **Diagnosis**: variable objetivo (0=No cáncer, 1=Cáncer)

**Dato médico importante 🧬:** Los factores de riesgo de cáncer se dividen en modificables (tabaquismo, dieta, ejercicio) y no modificables (genética, edad). Los modelos de ML permiten ponderar automáticamente el peso relativo de cada factor — algo que los modelos de riesgo clínico tradicionales (como el índice de Framingham para cardiopatías) hacen manualmente con ecuaciones fijas.


In [ ]:
import kagglehub
import os

# Descargamos el dataset de predicción de cáncer
path = kagglehub.dataset_download("rabieelkharoua/cancer-prediction-dataset")
print("Ruta al dataset:", path)
print(os.listdir(path))

In [ ]:
df = pd.read_csv(os.path.join(path, "The_Cancer_data_1500_V2.csv"))
df

In [ ]:
# Resumen del DataFrame: tipos de datos, valores nulos, memoria
df.info()

In [ ]:
# Distribución de la variable objetivo: ¿cuántos tienen cáncer vs. no?
class_counts = df['Diagnosis'].value_counts()

plt.figure(figsize=(7, 6))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title('Frecuencia de Diagnóstico de Cáncer')
plt.xlabel('Diagnóstico (0=No, 1=Sí)')
plt.ylabel('Número de pacientes')
plt.show()

print("Proporción:", (class_counts / len(df)).round(3))

---

## ⚙️ Preprocesamiento

### Label Encoding
Las variables categóricas de texto (como "Sí"/"No" o "Masculino"/"Femenino") necesitan convertirse a números para que los modelos puedan procesarlas.


In [ ]:
# Convertimos todas las columnas de texto a números
# LabelEncoder asigna un número entero a cada categoría
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [ ]:
# Eliminamos filas con valores nulos
df = df.dropna()
print("Filas después de dropna:", len(df))

In [ ]:
# Separamos features (X) y target (y)
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

print("Features:", X.shape)
print("Target:", y.value_counts())

---

## 🌲 Random Forest con SMOTE

### ¿Qué es SMOTE?

**SMOTE** (Synthetic Minority Over-sampling Technique) crea muestras sintéticas de la clase minoritaria interpolando entre ejemplos reales existentes.

**¿Por qué es necesario en medicina?** Los datasets médicos suelen estar desbalanceados — por ejemplo, si solo el 20% de los pacientes tiene cáncer, un modelo que siempre diga "no cáncer" tendría 80% de accuracy sin aprender nada. SMOTE equilibra las clases para que el modelo aprenda a detectar la condición rara.

> **Precaución ⚠️:** SMOTE solo se aplica al conjunto de *entrenamiento*, nunca al de prueba. Si aplicamos SMOTE al test, estaríamos evaluando el modelo con datos artificiales, lo que daría métricas infladas artificialmente.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Distribución en train antes de SMOTE:")
print(y_train.value_counts())

In [ ]:
# Aplicamos SMOTE para balancear las clases en el conjunto de entrenamiento
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Distribución en train DESPUÉS de SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

In [ ]:
# Hiperparámetros optimizados del Random Forest
# (en un ejercicio real, estos se obtienen con GridSearchCV o búsqueda aleatoria)
optimal_depth      = 24    # max_depth: profundidad máxima de cada árbol
optimal_estimator  = 200   # n_estimators: número de árboles en el bosque

model_rf = RandomForestClassifier(n_estimators=optimal_estimator,
                                  max_depth=optimal_depth,
                                  random_state=42)
model_rf.fit(X_train_resampled, y_train_resampled)
print("Random Forest entrenado.")

In [ ]:
y_pred_rf = model_rf.predict(X_test)

# Métricas de evaluación para clasificación binaria
accuracy  = accuracy_score(y_test, y_pred_rf)
precision = precision_score(y_test, y_pred_rf)
recall    = recall_score(y_test, y_pred_rf)
f1        = f1_score(y_test, y_pred_rf)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}  (de todos los que dije que tienen cáncer, ¿cuántos sí tenían?)")
print(f"Recall:    {recall:.4f}  (de todos los que SÍ tienen cáncer, ¿cuántos detecté?)")
print(f"F1 Score:  {f1:.4f}  (media armónica de precision y recall)")

---

## 🔍 Interpretabilidad con SHAP

**SHAP** (SHapley Additive exPlanations) usa la teoría de juegos cooperativos (valores de Shapley) para asignar a cada característica su contribución *justa* a la predicción de un modelo.

**¿Por qué importa la interpretabilidad en medicina?**
- Regulación: la FDA y la EMA exigen explicabilidad para dispositivos de diagnóstico con IA.
- Confianza médica: un doctor necesita entender POR QUÉ el modelo dice "cáncer".
- Detección de sesgos: SHAP puede revelar si el modelo usa variables inapropiadas (como raza o código postal).

El **SHAP summary plot** muestra:
- **Eje X:** valor SHAP (impacto en la predicción)
- **Color:** valor de la característica (rojo = alto, azul = bajo)
- **Eje Y:** características ordenadas por importancia


In [ ]:
import shap

# TreeExplainer: optimizado para modelos basados en árboles (RF, XGBoost, LightGBM)
explainer_rf   = shap.TreeExplainer(model_rf)
shap_values_rf = explainer_rf.shap_values(X_test)  # Calcula SHAP para cada muestra

# Summary plot: visión global de la importancia de características
shap.summary_plot(shap_values_rf, X_test)

---

## ⚡ XGBoost con Optimización de Hiperparámetros

**XGBoost** (eXtreme Gradient Boosting) es un algoritmo de ensemble que construye árboles *secuencialmente*, donde cada árbol nuevo corrige los errores del anterior.

**Diferencia con Random Forest:**
- RF: árboles en paralelo, independientes, combinados por votación
- XGBoost: árboles en serie, cada uno corrige al anterior (boosting)

**Parámetros importantes:**
| Parámetro | Descripción |
|---|---|
| `max_depth` | Profundidad máxima de cada árbol |
| `n_estimators` | Número de árboles (iteraciones de boosting) |
| `learning_rate` | Qué tanto "aprende" cada árbol nuevo |
| `subsample` | Fracción de muestras usadas por árbol |
| `colsample_bytree` | Fracción de features usados por árbol |
| `gamma` | Regularización: ganancia mínima para hacer split |
| `reg_lambda` | Regularización L2 en los pesos |

**GridSearchCV:** Prueba exhaustivamente todas las combinaciones de hiperparámetros y elige la mejor usando validación cruzada.


In [ ]:
df = pd.read_csv(os.path.join(path, "The_Cancer_data_1500_V2.csv"))

# Re-codificamos variables categóricas
label_encoders = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [ ]:
target_variable = 'Diagnosis'
X = df.drop(columns=[target_variable])
y = df[target_variable]

In [ ]:
print("Distribución de clases antes de SMOTE:")
print(y.value_counts())

In [ ]:
# División estratificada para mantener la proporción de clases en train y test
X_train, X_test_arr, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Aplicamos SMOTE al entrenamiento
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Escalamos los datos (XGBoost no lo requiere estrictamente, pero mejora la estabilidad)
scaler = StandardScaler()
X_train_res   = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test_arr)

print("Distribución después de SMOTE:")
print(pd.Series(y_train_res).value_counts())

In [ ]:
# GridSearch: probamos todas las combinaciones de max_depth y n_estimators
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'n_estimators': [100, 200, 300, 400, 500]
}

xgb_model = XGBClassifier(
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_lambda=1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# cv=3: validación cruzada con 3 particiones
# n_jobs=-1: usa todos los núcleos del CPU para acelerar
grid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_res, y_train_res)

In [ ]:
best_params = grid_search.best_params_
print(f"Mejor max_depth: {best_params['max_depth']}")
print(f"Mejor n_estimators: {best_params['n_estimators']}")

In [ ]:
# Entrenamos el modelo final con los mejores hiperparámetros
xgb_best_model = XGBClassifier(
    max_depth=best_params['max_depth'],
    n_estimators=best_params['n_estimators'],
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_lambda=1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_best_model.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_best_model.predict(X_test_scaled)

In [ ]:
def evaluate_model(y_test, y_pred, model_name):
    """Imprime un resumen completo de métricas de clasificación binaria"""
    acc      = accuracy_score(y_test, y_pred)
    prec     = precision_score(y_test, y_pred)
    rec      = recall_score(y_test, y_pred)
    f1       = f1_score(y_test, y_pred)
    roc_auc  = roc_auc_score(y_test, y_pred)

    print(f"\n{model_name} — Métricas:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")

evaluate_model(y_test, y_pred_xgb, "XGBoost (optimizado con GridSearch)")

In [ ]:
# Matriz de confusión del XGBoost
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_xgb),
            annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción')
plt.ylabel('Verdadero')
plt.title('Matriz de Confusión — XGBoost')
plt.show()

In [ ]:
# SHAP para XGBoost
X_test_df   = pd.DataFrame(X_test_scaled, columns=X.columns)
explainer2  = shap.Explainer(xgb_best_model)
shap_values2 = explainer2(X_test_df)
shap.summary_plot(shap_values2, X_test_df)

---

## 🔬 LIME — Explicaciones Locales

**LIME** (Local Interpretable Model-agnostic Explanations) explica la predicción para UNA instancia específica, no el modelo global.

**Idea:** LIME perturba los valores del ejemplo a explicar, observa cómo cambia la predicción, y ajusta un modelo lineal simple (que SÍ es interpretable) alrededor de ese punto.

**Caso de uso clínico:** "¿Por qué el modelo dice que ESTE paciente tiene 87% de probabilidad de cáncer?" LIME puede responder: "Principalmente por su historial de tabaquismo (40%) y su edad avanzada (25%)."


In [ ]:
from lime import lime_tabular

feature_names = X_test_df.columns.tolist()

# Inicializamos el explicador LIME con los datos de entrenamiento
explainer_lime = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train_res),
    feature_names=feature_names,
    class_names=['No Cáncer', 'Cáncer'],
    mode='classification'
)

# Elegimos la instancia 0 del test para explicar
i = 0
instancia = X_test_df.iloc[i]

exp = explainer_lime.explain_instance(
    data_row=instancia.values,
    predict_fn=xgb_best_model.predict_proba,
    num_features=5   # Mostramos las 5 características más influyentes
)

print(f"Explicación para el paciente #{i}")
print(f"Probabilidad predicha de cáncer: {exp.predict_proba[1]:.3f}")
print("\nFactores más influyentes:")
for feat, contrib in exp.as_list():
    sentido = "↑ aumenta riesgo" if contrib > 0 else "↓ reduce riesgo"
    print(f"  {feat}: {contrib:.4f}  {sentido}")

exp.show_in_notebook(show_all=True)